[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/070.%20The%20Minimum%20Spanning%20Tree%20Problem/P70-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 70. The Minimum Spanning Tree Problem

## Tier 2: The Classic Heuristic (Python Implementation)

### Key assumptions
- We can use greedy algorithms to find the optimal MST efficiently
- Kruskal's algorithm with Union-Find data structure provides optimal solution
- Time complexity is dominated by sorting edges: O(E log E)
- Union-Find with path compression enables efficient cycle detection

### Approach (step-by-step)
1. **Sort all edges** by weight in ascending order
2. **Initialize Union-Find** structure for cycle detection
3. **Iterate through sorted edges**: add each edge if it doesn't create a cycle
4. **Track selected edges** until we have n-1 edges
5. **Return the MST** with total cost

### What to look for in the results
- Step-by-step edge selection process
- Union-Find operations (find and union)
- Cycle detection in action
- Final MST structure and total cost
- Comparison with mathematical formulation (Tier 1)

### Concrete example (from the source)
We'll apply Kruskal's algorithm to the 8-city network:
- Cities: A, B, C, D, E, F, G, H
- Expected optimal cost: $36 million
- Expected edges: 7 (n-1 for 8 vertices)

### Why this Tier exists vs Tier 1
While Tier 1 provides exact optimization, Kruskal's algorithm offers:
- **Polynomial time complexity** vs exponential for MILP
- **Simple implementation** vs complex mathematical formulation
- **Guaranteed optimality** for standard MST problems
- **Practical applicability** for real-time applications

### Pros / Cons vs Tier 1
**Pros vs Tier 1:**
- Much faster (O(E log E) vs exponential)
- Simpler to implement and understand
- Guaranteed optimal solution for MST
- No need for specialized optimization solvers
- Works well for very large networks

**Cons vs Tier 1:**
- Limited to standard MST problems
- Cannot handle complex constraints easily
- Less flexible for multi-objective optimization
- No sensitivity analysis built-in

### When to use this Tier
- Standard MST problems without additional constraints
- Large-scale networks where MILP is too slow
- Real-time applications requiring fast solutions
- Educational purposes to understand greedy algorithms
- When you need guaranteed optimality with efficiency

In [1]:
from typing import Tuple, List, Dict, Set

# Import required packages for Kruskal's algorithm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Optional
import time
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Edge:
    """Represents an edge in the graph"""
    u: int
    v: int
    weight: float
    u_name: str
    v_name: str

@dataclass
class KruskalStep:
    """Represents a step in Kruskal's algorithm"""
    step: int
    edge: Edge
    action: str  # 'Selected' or 'Skipped'
    reason: str
    total_cost: float
    selected_edges: List[Edge]

@dataclass
class UnionFind:
    """Union-Find data structure with path compression and union by rank"""
    parent: List[int]
    rank: List[int]
    
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.rank = [0] * n
    
    def find(self, x: int) -> int:
        """Find the root of x with path compression"""
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    
    def union(self, x: int, y: int) -> bool:
        """Union the sets containing x and y. Return True if merged, False if already in same set"""
        px, py = self.find(x), self.find(y)
        if px == py:
            return False  # Already in the same set (would create a cycle)
        
        # Union by rank
        if self.rank[px] < self.rank[py]:
            self.parent[py] = px
        elif self.rank[px] > self.rank[py]:
            self.parent[px] = py
        else:
            self.parent[py] = px
            self.rank[px] += 1
        
        return True

In [ ]:
def create_8_city_network() -> Tuple[List[str], List[Edge]]:
    """Create the 8-city network from the problem statement"""
    cities = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
    
    edges = [
        Edge(0, 1, 12, 'A', 'B'),
        Edge(0, 2, 8, 'A', 'C'),
        Edge(0, 3, 6, 'A', 'D'),
        Edge(1, 2, 5, 'B', 'C'),
        Edge(1, 7, 7, 'B', 'H'),
        Edge(2, 3, 9, 'C', 'D'),
        Edge(3, 4, 4, 'D', 'E'),
        Edge(4, 6, 3, 'E', 'G'),
        Edge(5, 6, 5, 'F', 'G'),
        Edge(5, 7, 6, 'F', 'H'),
        Edge(6, 7, 8, 'G', 'H')
    ]
    
    return cities, edges

def kruskal_mst_with_steps(cities: List[str], edges: List[Edge]) -> Tuple[List[Edge], float, List[KruskalStep]]:
    """Kruskal's algorithm with detailed step tracking"""
    print("🔍 KRUSKAL'S ALGORITHM FOR MINIMUM SPANNING TREE")
    print("="*50)
    
    # Sort edges by weight
    sorted_edges = sorted(edges, key=lambda e: e.weight)
    print(f"\n📊 Sorted edges by weight:")
    for i, edge in enumerate(sorted_edges, 1):
        print(f"   {i:2d}. {edge.u_name}-{edge.v_name}: ${edge.weight}M")
    
    # Initialize Union-Find
    uf = UnionFind(len(cities))
    selected_edges = []
    total_cost = 0
    steps = []
    
    print(f"\n🔄 Algorithm execution:")
    step_count = 0
    
    for edge in sorted_edges:
        step_count += 1
        
        # Check if adding this edge creates a cycle
        if uf.union(edge.u, edge.v):
            # Edge selected
            selected_edges.append(edge)
            total_cost += edge.weight
            action = "Selected"
            reason = "No cycle created"
            print(f"   Step {step_count:2d}: {edge.u_name}-{edge.v_name} SELECTED (${edge.weight}M) - Total: ${total_cost}M")
        else:
            # Edge skipped (would create cycle)
            action = "Skipped"
            reason = "Would create cycle"
            print(f"   Step {step_count:2d}: {edge.u_name}-{edge.v_name} SKIPPED (${edge.weight}M) - Would create cycle")
        
        # Record step
        step = KruskalStep(
            step=step_count,
            edge=edge,
            action=action,
            reason=reason,
            total_cost=total_cost,
            selected_edges=selected_edges.copy()
        )
        steps.append(step)
        
        # Stop if we have n-1 edges
        if len(selected_edges) == len(cities) - 1:
            break
    
    return selected_edges, total_cost, steps

In [ ]:
def visualize_kruskal_steps(cities: List[str], edges: List[Edge], steps: List[KruskalStep]):
    """Visualize the step-by-step execution of Kruskal's algorithm"""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Create positions for cities (circular layout)
    n = len(cities)
    angles = np.linspace(0, 2*np.pi, n, endpoint=False)
    positions = {city: (np.cos(angle), np.sin(angle)) for city, angle in zip(cities, angles)}
    
    # Plot 1: Initial network
    ax1 = axes[0, 0]
    ax1.set_title('Initial Network', fontsize=14, fontweight='bold')
    
    for edge in edges:
        pos1 = positions[edge.u_name]
        pos2 = positions[edge.v_name]
        ax1.plot([pos1[0], pos2[0]], [pos1[1], pos2[1]], 'gray', alpha=0.5, linewidth=1)
        ax1.text((pos1[0]+pos2[0])/2, (pos1[1]+pos2[1])/2, f'${edge.weight}M', 
                fontsize=8, ha='center', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    
    for city, pos in positions.items():
        ax1.plot(pos[0], pos[1], 'o', markersize=10, markerfacecolor='lightblue', 
                markeredgecolor='navy', markeredgewidth=2)
        ax1.text(pos[0], pos[1]-0.15, city, fontsize=12, ha='center', fontweight='bold')
    
    ax1.set_xlim(-1.5, 1.5)
    ax1.set_ylim(-1.5, 1.5)
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Final MST
    ax2 = axes[0, 1]
    selected_edges = [step.edge for step in steps if step.action == 'Selected']
    total_cost = sum(edge.weight for edge in selected_edges)
    ax2.set_title(f'Final MST\nTotal Cost: ${total_cost}M', fontsize=14, fontweight='bold')
    
    # Plot all edges in light gray
    for edge in edges:
        pos1 = positions[edge.u_name]
        pos2 = positions[edge.v_name]
        ax2.plot([pos1[0], pos2[0]], [pos1[1], pos2[1]], 'lightgray', alpha=0.3, linewidth=1)
    
    # Plot selected edges in green
    for edge in selected_edges:
        pos1 = positions[edge.u_name]
        pos2 = positions[edge.v_name]
        ax2.plot([pos1[0], pos2[0]], [pos1[1], pos2[1]], 'green', linewidth=3)
        ax2.text((pos1[0]+pos2[0])/2, (pos1[1]+pos2[1])/2, f'${edge.weight}M', 
                fontsize=8, ha='center', bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.8))
    
    for city, pos in positions.items():
        ax2.plot(pos[0], pos[1], 'o', markersize=10, markerfacecolor='lightgreen', 
                markeredgecolor='darkgreen', markeredgewidth=2)
        ax2.text(pos[0], pos[1]-0.15, city, fontsize=12, ha='center', fontweight='bold')
    
    ax2.set_xlim(-1.5, 1.5)
    ax2.set_ylim(-1.5, 1.5)
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Edge selection timeline
    ax3 = axes[1, 0]
    ax3.set_title('Edge Selection Timeline', fontsize=14, fontweight='bold')
    
    selected_steps = [step for step in steps if step.action == 'Selected']
    step_numbers = [step.step for step in selected_steps]
    costs = [step.edge.weight for step in selected_steps]
    cumulative_costs = [step.total_cost for step in selected_steps]
    
    ax3.bar(step_numbers, costs, alpha=0.7, color='green', label='Edge Cost')
    ax3.plot(step_numbers, cumulative_costs, 'ro-', linewidth=2, markersize=8, label='Cumulative Cost')
    ax3.set_xlabel('Step Number')
    ax3.set_ylabel('Cost ($M)')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Algorithm statistics
    ax4 = axes[1, 1]
    ax4.set_title('Algorithm Statistics', fontsize=14, fontweight='bold')
    
    # Create statistics
    total_edges = len(edges)
    selected_count = len(selected_edges)
    skipped_count = len([step for step in steps if step.action == 'Skipped'])
    
    stats_data = [
        ('Total Edges', total_edges, 'blue'),
        ('Selected Edges', selected_count, 'green'),
        ('Skipped Edges', skipped_count, 'red'),
        ('Total Cost ($M)', total_cost, 'orange')
    ]
    
    y_pos = range(len(stats_data))
    values = [stat[1] for stat in stats_data]
    colors = [stat[2] for stat in stats_data]
    labels = [stat[0] for stat in stats_data]
    
    bars = ax4.barh(y_pos, values, color=colors, alpha=0.7)
    ax4.set_yticks(y_pos)
    ax4.set_yticklabels(labels)
    ax4.set_xlabel('Value')
    
    # Add value labels on bars
    for i, (bar, value) in enumerate(zip(bars, values)):
        ax4.text(value + 0.1, bar.get_y() + bar.get_height()/2, str(value), 
                va='center', fontweight='bold')
    
    ax4.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()

In [ ]:
def analyze_kruskal_performance(cities: List[str], edges: List[Edge], selected_edges: List[Edge], 
                               total_cost: float, steps: List[KruskalStep]):
    """Analyze the performance of Kruskal's algorithm"""
    print("="*60)
    print("KRUSKAL'S ALGORITHM - PERFORMANCE ANALYSIS")
    print("="*60)
    
    # Final results
    print(f"\n🎯 FINAL RESULTS:")
    print(f"   Total edges processed: {len(steps)}")
    print(f"   Edges selected: {len(selected_edges)}")
    print(f"   Edges skipped: {len([s for s in steps if s.action == 'Skipped'])}")
    print(f"   Total cost: ${total_cost} million")
    print(f"   Algorithm steps: {len(steps)}")
    
    print(f"\n📊 SELECTED EDGES:")
    for i, edge in enumerate(selected_edges, 1):
        print(f"   {i}. {edge.u_name}-{edge.v_name}: ${edge.weight} million")
    
    # Comparison with optimal
    print(f"\n🏆 OPTIMALITY ANALYSIS:")
    optimal_cost = 36  # From problem statement
    if total_cost == optimal_cost:
        print(f"   ✅ OPTIMAL SOLUTION FOUND!")
        print(f"   Algorithm cost: ${total_cost} million")
        print(f"   Expected optimal: ${optimal_cost} million")
        print(f"   Optimality gap: 0.00%")
    else:
        gap = ((total_cost - optimal_cost) / optimal_cost) * 100
        print(f"   ⚠️  Suboptimal solution found")
        print(f"   Algorithm cost: ${total_cost} million")
        print(f"   Expected optimal: ${optimal_cost} million")
        print(f"   Optimality gap: {gap:.2f}%")
    
    # Algorithm efficiency
    print(f"\n⚡ ALGORITHM EFFICIENCY:")
    print(f"   Time complexity: O(E log E)")
    print(f"   Space complexity: O(V)")
    print(f"   Number of vertices: {len(cities)}")
    print(f"   Number of edges: {len(edges)}")
    print(f"   Union-Find operations: {len(steps) * 2}")  # Each step does find + union
    
    # Edge rejection analysis
    print(f"\n🔄 EDGE REJECTION ANALYSIS:")
    skipped_edges = [step for step in steps if step.action == 'Skipped']
    print(f"   Edges rejected: {len(skipped_edges)}")
    print(f"   Rejection rate: {len(skipped_edges)/len(steps)*100:.1f}%")
    
    if skipped_edges:
        print(f"\n   Rejected edges (by weight):")
        for step in skipped_edges:
            print(f"      {step.edge.u_name}-{step.edge.v_name}: ${step.edge.weight}M ({step.reason})")
    
    # Comparison with other methods
    print(f"\n🔍 METHOD COMPARISON:")
    print(f"   Mathematical formulation: Exponential time, optimal")
    print(f"   Kruskal's algorithm: O(E log E), optimal")
    print(f"   Prim's algorithm: O(E log V), optimal")
    print(f"   Genetic algorithm: O(G × P × E), near-optimal")
    
    # When to use Kruskal's
    print(f"\n💡 WHEN TO USE KRUSKAL'S ALGORITHM:")
    print(f"   ✓ Standard MST problems")
    print(f"   ✓ Sparse graphs (E ≈ V)")
    print(f"   ✓ When you need all edges sorted")
    print(f"   ✓ Easy to implement and understand")
    print(f"   ✓ When edge list is readily available")
    
    # Limitations
    print(f"\n⚠️  LIMITATIONS:")
    print(f"   ✗ Not suitable for dense graphs (Prim's is better)")
    print(f"   ✗ Cannot handle negative edge weights")
    print(f"   ✗ Limited to standard MST problems")
    print(f"   ✗ No built-in support for constraints")
    
    return selected_edges, total_cost

In [ ]:
# Main execution - run Kruskal's algorithm
print("🔍 EXECUTING KRUSKAL'S ALGORITHM FOR MINIMUM SPANNING TREE")
print("="*60)

# Create the network
cities, edges = create_8_city_network()

print(f"\n📊 PROBLEM SETUP:")
print(f"   Cities: {cities}")
print(f"   Number of vertices: {len(cities)}")
print(f"   Number of edges: {len(edges)}")
print(f"   Expected optimal cost: $36 million")

# Run Kruskal's algorithm with step tracking
start_time = time.time()
selected_edges, total_cost, steps = kruskal_mst_with_steps(cities, edges)
end_time = time.time()

print(f"\n⏱️  EXECUTION TIME: {(end_time - start_time)*1000:.2f} milliseconds")

# Analyze performance
analyze_kruskal_performance(cities, edges, selected_edges, total_cost, steps)

# Visualize the algorithm execution
print(f"\n🎨 ALGORITHM VISUALIZATION")
visualize_kruskal_steps(cities, edges, steps)

🔍 EXECUTING KRUSKAL'S ALGORITHM FOR MINIMUM SPANNING TREE

📊 PROBLEM SETUP:
   Cities: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
   Number of vertices: 8
   Number of edges: 11
   Expected optimal cost: $36 million
🔍 KRUSKAL'S ALGORITHM FOR MINIMUM SPANNING TREE

📊 Sorted edges by weight:
    1. E-G: $3M
    2. D-E: $4M
    3. B-C: $5M
    4. F-G: $5M
    5. A-D: $6M
    6. F-H: $6M
    7. B-H: $7M
    8. A-C: $8M
    9. G-H: $8M
   10. C-D: $9M
   11. A-B: $12M

🔄 Algorithm execution:
   Step  1: E-G SELECTED ($3M) - Total: $3M
   Step  2: D-E SELECTED ($4M) - Total: $7M
   Step  3: B-C SELECTED ($5M) - Total: $12M
   Step  4: F-G SELECTED ($5M) - Total: $17M
   Step  5: A-D SELECTED ($6M) - Total: $23M
   Step  6: F-H SELECTED ($6M) - Total: $29M
   Step  7: B-H SELECTED ($7M) - Total: $36M

⏱️  EXECUTION TIME: 0.07 milliseconds
KRUSKAL'S ALGORITHM - PERFORMANCE ANALYSIS

🎯 FINAL RESULTS:
   Total edges processed: 7
   Edges selected: 7
   Edges skipped: 0
   Total cost: $36 million
 

In [ ]:
try:
    # Additional analysis: Union-Find operations visualization
    def visualize_union_find_operations(cities: List[str], edges: List[Edge], steps: List[KruskalStep]):
        """Visualize Union-Find operations during Kruskal's algorithm"""
        print("\n" + "="*60)
        print("UNION-FIND OPERATIONS ANALYSIS")
        print("="*60)
        
        # Track Union-Find state after each operation
        uf_states = []
        uf = UnionFind(len(cities))
        
        for step in steps:
            if step.action == 'Selected':
                # Save current state
                current_state = {
                    'step': step.step,
                    'edge': f"{step.edge.u_name}-{step.edge.v_name}",
                    'parent': uf.parent.copy(),
                    'rank': uf.rank.copy()
                }
                uf_states.append(current_state)
                
                # Perform the union
                uf.union(step.edge.u, step.edge.v)
        
        print(f"\n📊 UNION-FIND STATE EVOLUTION:")
        for state in uf_states:
            print(f"\n   After step {state['step']} ({state['edge']}):")
            print(f"      Parent array: {state['parent']}")
            print(f"      Rank array:   {state['rank']}")
            
            # Show connected components
            components = {}
            for i, parent in enumerate(state['parent']):
                root = uf.find(i)  # Use current UF to find final root
                if root not in components:
                    components[root] = []
                components[root].append(cities[i])
            
            print(f"      Components: {{' | '.join(comp) for comp in components.values()}}}")
        
        # Final state
        print(f"\n🎯 FINAL UNION-FIND STATE:")
        print(f"   Final parent array: {uf.parent}")
        print(f"   Final rank array:   {uf.rank}")
        
        # Performance metrics
        print(f"\n⚡ UNION-FIND PERFORMANCE:")
        total_find_ops = len(steps) * 2  # Each union does 2 finds
        total_union_ops = len([s for s in steps if s.action == 'Selected'])
        
        print(f"   Total find operations: {total_find_ops}")
        print(f"   Total union operations: {total_union_ops}")
        print(f"   Path compression effectiveness: High (inverse Ackermann)")
        print(f"   Union by rank effectiveness: Prevents tall trees")
        
    # Run Union-Find analysis
    visualize_union_find_operations(cities, edges, steps)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: f-string: single '}' is not allowed (<string>, line 40)...]

In [ ]:
# Edge rejection analysis
def edge_rejection_analysis():
    """Detailed analysis of rejected edges and their reasons"""
    print("\n" + "="*60)
    print("EDGE REJECTION ANALYSIS")
    print("="*60)
    
    skipped_steps = [step for step in steps if step.action == 'Skipped']
    
    print(f"\n📊 REJECTION STATISTICS:")
    print(f"   Total edges considered: {len(steps)}")
    print(f"   Edges selected: {len([s for s in steps if s.action == 'Selected'])}")
    print(f"   Edges rejected: {len(skipped_steps)}")
    print(f"   Rejection rate: {len(skipped_steps)/len(steps)*100:.1f}%")
    
    if skipped_steps:
        print(f"\n🔄 REJECTED EDGES (in order of consideration):")
        for step in skipped_steps:
            print(f"   Step {step.step}: {step.edge.u_name}-{step.edge.v_name} rejected at cost ${step.edge.weight}M")

# Run edge rejection analysis
edge_rejection_analysis()


EDGE REJECTION ANALYSIS

📊 REJECTION STATISTICS:
   Total edges considered: 7
   Edges selected: 7
   Edges rejected: 0
   Rejection rate: 0.0%
